## **EVALUACIÓN ZERO-SHOT**

En este fichero se evalúa el rendimiento de todos los modelos de SAM estudiados en el proyecto utilizando el conjunto de datos KVASIR-SEG.

In [ ]:
from utils.segmentation_quality_metrics import get_boundary, boundary_iou, resize_for_hausdorff, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_zero_shot, measure_inference_sam3_prompt_zero_shot
from ultralytics import SAM
from ultralytics.models.sam import SAM3SemanticPredictor
import numpy as np
import os
import cv2
import pandas as pd
import json
import time
import sys
import os


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"


**SAM 1 BASE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.5770
Mean F1 Score: 0.6622
Mean Recall: 0.7023
Mean Precision: 0.8565
Mean Dice: 0.6622
Mean Specificity: 0.9300
Mean F2: 0.6666
Mean Pixel Accuracy: 0.8957
Mean Boundary IoU: 0.0393
Mean Hausdorff-95: 136.4489
Mean Latency (ms): 222.20
Mean VRAM Usage (MB): 0.70


**SAM 1 LARGE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.6368
Mean F1 Score: 0.7128
Mean Recall: 0.7192
Mean Precision: 0.9038
Mean Dice: 0.7128
Mean Specificity: 0.9632
Mean F2: 0.7037
Mean Pixel Accuracy: 0.9217
Mean Boundary IoU: 0.0492
Mean Hausdorff-95: 124.7846
Mean Latency (ms): 513.59
Mean VRAM Usage (MB): 1.53


**SAM 2 BASE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.6251
Mean F1 Score: 0.7062
Mean Recall: 0.8176
Mean Precision: 0.7997
Mean Dice: 0.7062
Mean Specificity: 0.8360
Mean F2: 0.7261
Mean Pixel Accuracy: 0.8362
Mean Boundary IoU: 0.0557
Mean Hausdorff-95: 140.7203
Mean Latency (ms): 217.09
Mean VRAM Usage (MB): 0.75


**SAM 2 LARGE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.6432
Mean F1 Score: 0.7222
Mean Recall: 0.8451
Mean Precision: 0.7908
Mean Dice: 0.7222
Mean Specificity: 0.8279
Mean F2: 0.7447
Mean Pixel Accuracy: 0.8313
Mean Boundary IoU: 0.0643
Mean Hausdorff-95: 141.7166
Mean Latency (ms): 390.29
Mean VRAM Usage (MB): 1.31


**SAM 2.1 BASE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2.1_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.6374
Mean F1 Score: 0.7164
Mean Recall: 0.8188
Mean Precision: 0.8105
Mean Dice: 0.7164
Mean Specificity: 0.8488
Mean F2: 0.7336
Mean Pixel Accuracy: 0.8471
Mean Boundary IoU: 0.0564
Mean Hausdorff-95: 138.5723
Mean Latency (ms): 209.98
Mean VRAM Usage (MB): 0.75


**SAM 2.1 LARGE**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2.1_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.6663
Mean F1 Score: 0.7450
Mean Recall: 0.8408
Mean Precision: 0.8189
Mean Dice: 0.7450
Mean Specificity: 0.8583
Mean F2: 0.7606
Mean Pixel Accuracy: 0.8565
Mean Boundary IoU: 0.0651
Mean Hausdorff-95: 134.3480
Mean Latency (ms): 379.47
Mean VRAM Usage (MB): 1.31


## **SAM 3**

In [ ]:
def evaluate_model(dataset):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_zero_shot(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam3"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

SAM 3 USANDO PROMPTS PARA SEGMENTAR

In [ ]:
def evaluate_model(dataset, text_prompt="polyp"):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    overrides = dict(
        conf=0.01,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )

    predictor = SAM3SemanticPredictor(overrides=overrides)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        results, latency, vram = measure_inference_sam3_prompt_zero_shot(predictor, img_path, text_prompt)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores    = results[0].probs
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[0].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam3_prompt"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_kvasir.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Ultralytics 8.4.23  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\Kvasir-SEG\images\cju0qkwl35piu0993l0dewei2.jpg: 644x644 (no detections), 77.6ms
Speed: 3.3ms preprocess, 77.6ms inference, 0.9ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM1-zero-shot\runs\segment\predict16
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\Kvasir-SEG\images\cju0qoxqj9q6s0835b43399p4.jpg: 644x644 (no detections), 77.1ms
Speed: 3.4ms preprocess, 77.1ms inference, 0.9ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM1-zero-shot\runs\segment\predict16
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\Kvasir-SEG\images

In [83]:
import inspect

overrides = dict(
        conf=0.01,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )

predictor = SAM3SemanticPredictor(overrides=overrides)
# print(inspect.getsource(SAM3SemanticPredictor))

import torch.nn.functional as F
from ultralytics.utils import ops

original_postprocess = predictor.postprocess

def debug_postprocess(preds, img, orig_imgs):
    pred_logits = preds["pred_logits"]
    pred_scores = pred_logits.sigmoid()
    presence_score = preds["presence_logit_dec"].sigmoid().unsqueeze(1)
    combined = (pred_scores * presence_score).squeeze(-1)
    print("Max score:", combined.max().item())
    print("Mean score:", combined.mean().item())
    return original_postprocess(preds, img, orig_imgs)

predictor.postprocess = debug_postprocess

img_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\images\\cju0qkwl35piu0993l0dewei2.jpg"
predictor.set_image(img_path)
predictor.model.set_classes(text=["polyp"])
predictor.prompts["text"] = ["polyp"]
results = predictor()

Ultralytics 8.4.23  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

Max score: 0.0002260225883219391
Mean score: 4.4553878979058936e-05
image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\Kvasir-SEG\images\cju0qkwl35piu0993l0dewei2.jpg: 644x644 (no detections), 77.4ms
Speed: 3.2ms preprocess, 77.4ms inference, 1.6ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM1-zero-shot\runs\segment\predict17
